In [ ]:
#Проверка доступности карточки
import torch
import numpy as np

print(torch.cuda.is_available())

In [ ]:
import pickle
import numpy as np
from skimage import io

from tqdm import tqdm, tqdm_notebook
from PIL import Image
from pathlib import Path

from torchvision import transforms
from torchvision.transforms import v2

import torchsummary

from multiprocessing.pool import ThreadPool
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from matplotlib import colors, pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings(action='ignore', category=DeprecationWarning)

In [ ]:
#Загрузка изначального датасета и формирование локального
import pandas as pd
from pathlib import Path
from tqdm import tqdm

dataset_root = Path('/kaggle/input/architectural-styles-dataset')

filepaths = []
labels = []

for filepath in tqdm(dataset_root.rglob('*')):
    
    if filepath.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        
        label = filepath.parent.name
        
        filepaths.append(str(filepath))
        labels.append(label)

df = pd.DataFrame({
    'filepath': filepaths,
    'label': labels
})

print(df.head())
df.to_csv('pathes_and_labels.csv', index=False)

In [ ]:
#Описание датасета для подготовки данных
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

class ArchDataset(Dataset):
    def __init__(self, files, label_encoder, mode):
        super().__init__()
        # список файлов для загрузки
        self.files = files
        # режим работы
        self.mode = mode
        self.label_encoder = label_encoder
        self.len_ = len(self.files)

        common_transforms = [
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ]

        if self.mode == 'train':
                self.transform = A.Compose([
                    A.Resize(512, 512),
                    A.RandomCrop(448, 448),

                    A.HorizontalFlip(p=0.5),
                    A.Perspective(scale=(0.05, 0.1), p=0.5),

                    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
                    A.ToGray(p=0.1),

                    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),

                    A.CoarseDropout(max_holes=8, max_height=30, max_width=30, p=0.3),
                ] + common_transforms)
        else:
          self.transform = A.Compose([
                    A.Resize(448, 448),
                ] + common_transforms)

    def __len__(self):
        return self.len_

    def __getitem__(self, index):
        image_pil = self.load_image(self.files[index])

        image_np = np.array(image_pil)

        augmented = self.transform(image=image_np)
        x = augmented['image']

        if self.mode == 'test':
            return x
        else:
            lbl = self.files[index]
            lbl = Path(lbl)
            label_str = lbl.parent.name
            y = self.label_encoder.transform([label_str]).item()
            return x, y


    def load_image(self, file):
        image = Image.open(file).convert('RGB')
        return image

In [ ]:
#Подготовка кросс-валидации
df = pd.read_csv('pathes_and_labels.csv')

X = df['filepath']
y = df['label']

train_val_files, test_files, train_val_labels, test_labels = train_test_split(
    X, 
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

train_files, val_files, train_labels, val_labels = train_test_split(
    train_val_files,
    train_val_labels,
    test_size=0.2,
    stratify=train_val_labels,
    random_state=42,
    shuffle=True
)

label_encoder = LabelEncoder()
label_encoder.fit(y)

train_files = train_files.reset_index(drop=True)
val_files = val_files.reset_index(drop=True)
train_labels = train_labels.reset_index(drop=True)
val_labels = val_labels.reset_index(drop=True)

In [ ]:
#Создание лоадеров
train_dataset = ArchDataset(train_files, label_encoder, 'train')
val_dataset = ArchDataset(val_files, label_encoder, mode='val')
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
#Импорт резнет-50
import torch.optim as optim
from torchvision import models

def get_resnet_50(num_classes):
    model = models.resnet50(weights='DEFAULT')

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )

    return model

n_classes = len(set(labels))
model = get_resnet_50(n_classes)
model = model.to('cuda')

In [ ]:
#Параметры для обучения последнего слоя
criterion = nn.CrossEntropyLoss()
optimizer_for_last_layer = torch.optim.Adam(model.fc.parameters(), lr = 1e-3)
num_epochs = 15

In [ ]:
#эпоха для валидации
def eval_epoch(model, val_loader, criterion, device):
    model.eval()

    val_batch_losses = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            val_batch_losses.append(loss.item())

    val_f1 = f1_score(all_labels, all_preds, average='macro')

    return val_batch_losses, sum(val_batch_losses) / len(val_batch_losses), val_f1

In [ ]:
#Эпоха для трейна
from PIL import Image

def train_epoch(model, train_loader, criterion, optimizer, device):
  model.train()

  batch_losses = []

  for inputs, labels in train_loader:
    inputs = inputs.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    batch_losses.append(loss.item())

  return batch_losses, sum(batch_losses) / len(batch_losses)

In [ ]:
#Обучение последнего слоя
from sklearn.metrics import f1_score

losses_per_epoch = []
batch_losses_per_epoch = []
val_losses_per_epoch = []
val_batch_losses_per_epoch = []
for epoch in tqdm(range(num_epochs)):
  batch_losses, loss_per_epoch = train_epoch(model,
                                             train_loader,
                                             criterion,
                                             optimizer_for_last_layer,
                                             'cuda'
                                             )
  losses_per_epoch.append(loss_per_epoch)
  batch_losses_per_epoch.append(batch_losses)
  batch_losses, loss_per_epoch, val_f1 = eval_epoch(model,
                                            val_loader,
                                            criterion,
                                            'cuda'
                                            )
  print(f'epoch number {epoch} f1: {val_f1}')
  val_losses_per_epoch.append(loss_per_epoch)
  val_batch_losses_per_epoch.append(batch_losses)

In [ ]:
#отрисовка лоссов по трейну
plt.figure(figsize=(10,5))
plt.plot(losses_per_epoch, color='blue')
plt.title('Losses per epoch')
plt.show()

import math

n_epochs = len(losses_per_epoch)

cols = 2
rows = math.ceil(n_epochs / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
fig.suptitle('Loss по трейну по батчам', fontsize=16)

axes_flat = axes.flatten()

for i in range(n_epochs):
    current_epoch_losses = batch_losses_per_epoch[i]

    ax = axes_flat[i]
    ax.plot(current_epoch_losses, color='blue', alpha=0.7, linewidth=1)

    ax.set_title(f"Эпоха {i+1}")
    ax.set_xlabel("Номер батча")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)

for i in range(n_epochs, rows * cols):
    fig.delaxes(axes_flat[i])

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

In [ ]:
#Отрисовка лоссов по валидации
plt.figure(figsize=(10,5))
plt.plot(val_losses_per_epoch, color='blue')
plt.title('Losses per epoch')
plt.show()

n_epochs = len(val_losses_per_epoch)

cols = 2
rows = math.ceil(n_epochs / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
fig.suptitle('Loss по валидации по батчам', fontsize=16)

axes_flat = axes.flatten()

for i in range(n_epochs):
    current_epoch_losses = val_batch_losses_per_epoch[i]

    ax = axes_flat[i]
    ax.plot(current_epoch_losses, color='blue', alpha=0.7, linewidth=1)

    ax.set_title(f"Эпоха {i+1}")
    ax.set_xlabel("Номер батча")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)

for i in range(n_epochs, rows * cols):
    fig.delaxes(axes_flat[i])

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

In [ ]:
#Параметры для обучения всей сети
optimizer_for_whole_net = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer_for_whole_net, step_size=3, gamma=0.1)
num_epochs = 20

In [ ]:
#Обучение всей сети
for param in model.parameters():
    param.requires_grad = True

best_f1 = 0.0

losses_per_epoch = []
batch_losses_per_epoch = []
val_losses_per_epoch = []
val_batch_losses_per_epoch = []
for epoch in tqdm(range(num_epochs)):
    batch_losses, loss_per_epoch = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer_for_whole_net,
        'cuda')
    losses_per_epoch.append(loss_per_epoch)
    batch_losses_per_epoch.append(batch_losses)
    batch_losses, loss_per_epoch, val_f1 = eval_epoch(model,
                                                      val_loader,
                                                      criterion,
                                                      'cuda')
    val_losses_per_epoch.append(loss_per_epoch)
    val_batch_losses_per_epoch.append(batch_losses)
    print(f'epoch number {epoch + 1} f1: {val_f1}')


    scheduler.step()

    val_acc = val_losses_per_epoch[-1]
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_arch_resnet_50.pth")
        print("Model saved!")